# 02 — IDD Dataset Validation

Purpose:
- Validate Pascal VOC XML annotations.
- Check bounding-box coordinates.
- Identify invalid or empty annotations.
- Summarize dataset quality before YOLO label conversion.

In [2]:
# CELL 2: Imports and dataset paths
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd
import json

ROOT = Path("/workspace/IDD_Detection")
PROJECT = Path("/workspace/ITS_Project")

metadata = json.loads(
    (PROJECT / "configs" / "idd_metadata.json").read_text()
)

print("Dataset metadata loaded.")
print("Training images:", metadata["splits"]["train"])
print("Validation images:", metadata["splits"]["val"])
print("Test images:", metadata["splits"]["test"])

Dataset metadata loaded.
Training images: 31569
Validation images: 10225
Test images: 4794


In [3]:
# CELL 3: Prepare annotation lists
train_entries = [
    x.strip() for x in (ROOT / "train.txt").read_text().splitlines()
    if x.strip()
]
val_entries = [
    x.strip() for x in (ROOT / "val.txt").read_text().splitlines()
    if x.strip()
]

xml_files = {
    "train": [
        (ROOT / "Annotations" / x).with_suffix(".xml")
        for x in train_entries
    ],
    "val": [
        (ROOT / "Annotations" / x).with_suffix(".xml")
        for x in val_entries
    ],
}

print("Training XML paths:", len(xml_files["train"]))
print("Validation XML paths:", len(xml_files["val"]))

Training XML paths: 31569
Validation XML paths: 10225


In [4]:
# CELL 4: Validate XML structure and bounding-box coordinates
def validate_xml(xml_path):
    issues = []

    if not xml_path.is_file():
        return ["missing_xml"]

    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError:
        return ["malformed_xml"]

    objects = root.findall("object")
    if not objects:
        issues.append("no_objects")

    for obj in objects:
        box = obj.find("bndbox")

        if box is None:
            issues.append("missing_bndbox")
            continue

        try:
            xmin = float(box.findtext("xmin"))
            ymin = float(box.findtext("ymin"))
            xmax = float(box.findtext("xmax"))
            ymax = float(box.findtext("ymax"))
        except (TypeError, ValueError):
            issues.append("invalid_coordinates")
            continue

        if xmax <= xmin or ymax <= ymin:
            issues.append("nonpositive_box")

    return issues

print("Validation function ready.")

Validation function ready.


In [5]:
# CELL 5: Run validation on train and validation annotations
validation_results = []

for split, paths in xml_files.items():
    for path in paths:
        issues = validate_xml(path)
        validation_results.append({
            "split": split,
            "xml_path": str(path.relative_to(ROOT)),
            "issue_count": len(issues),
            "issues": ", ".join(issues)
        })

validation_df = pd.DataFrame(validation_results)

print("Annotations checked:", len(validation_df))
print("Annotations with issues:", (validation_df["issue_count"] > 0).sum())
display(validation_df[validation_df["issue_count"] > 0].head(20))

Annotations checked: 41794
Annotations with issues: 1


,split,xml_path,issue_count,issues
29410,train,Annotations/highquality_16k/HYD-2018-04-26_13-...,1,nonpositive_box


In [6]:
# CELL 6: Summarize annotation validation results
issue_summary = (
    validation_df[validation_df["issue_count"] > 0]
    .assign(issue=lambda df: df["issues"].str.split(", "))
    .explode("issue")
    .groupby(["split", "issue"])
    .size()
    .reset_index(name="count")
)

display(issue_summary)

,split,issue,count
0,train,nonpositive_box,1


In [7]:
# CELL 7: Annotation quality summary
quality_summary = validation_df.groupby("split").agg(
    total_annotations=("xml_path", "count"),
    problematic_annotations=("issue_count", lambda x: (x > 0).sum())
)

quality_summary["valid_annotations"] = (
    quality_summary["total_annotations"]
    - quality_summary["problematic_annotations"]
)

display(quality_summary)

,total_annotations,problematic_annotations,valid_annotations
split,,,
train,31569,1,31568
val,10225,0,10225


In [8]:
# CELL 8: Check image dimensions recorded in XML
dimension_results = []

for split, paths in xml_files.items():
    for path in paths:
        if not path.is_file():
            continue

        try:
            root = ET.parse(path).getroot()
            size = root.find("size")

            if size is None:
                dimension_results.append({
                    "split": split,
                    "xml_path": str(path.relative_to(ROOT)),
                    "width": None,
                    "height": None
                })
                continue

            dimension_results.append({
                "split": split,
                "xml_path": str(path.relative_to(ROOT)),
                "width": int(size.findtext("width")),
                "height": int(size.findtext("height"))
            })
        except (ET.ParseError, TypeError, ValueError):
            continue

dimensions_df = pd.DataFrame(dimension_results)

print("XML dimensions inspected:", len(dimensions_df))
display(dimensions_df.groupby(["split", "width", "height"], dropna=False).size()
        .reset_index(name="image_count").head(20))

XML dimensions inspected: 41794


,split,width,height,image_count
0,train,1280,964,3431
1,train,1920,1080,26989
2,train,1921,1080,1149
3,val,1280,720,421
4,val,1280,964,1065
5,val,1920,1080,8535
6,val,1921,1080,204


In [9]:
# CELL 9: Save validation reports
REPORTS = PROJECT / "results" / "dataset_validation"
REPORTS.mkdir(parents=True, exist_ok=True)

validation_df.to_csv(REPORTS / "annotation_validation.csv", index=False)
issue_summary.to_csv(REPORTS / "issue_summary.csv", index=False)
quality_summary.to_csv(REPORTS / "quality_summary.csv")

print("Validation reports saved to:", REPORTS)

Validation reports saved to: /workspace/ITS_Project/results/dataset_validation


In [10]:
# CELL 10: Final dataset validation summary
print("DATASET VALIDATION COMPLETE")
print("-" * 40)

for split in ["train", "val"]:
    subset = validation_df[validation_df["split"] == split]
    total = len(subset)
    problematic = (subset["issue_count"] > 0).sum()

    print(f"{split.upper()}:")
    print(f"  XML files expected: {total:,}")
    print(f"  Files with issues:  {problematic:,}")
    print(f"  Files without issues: {total - problematic:,}")

print("\nTest annotations: unavailable")
print("Test-set detection metrics: cannot be calculated without labels.")

DATASET VALIDATION COMPLETE
----------------------------------------
TRAIN:
  XML files expected: 31,569
  Files with issues:  1
  Files without issues: 31,568
VAL:
  XML files expected: 10,225
  Files with issues:  0
  Files without issues: 10,225

Test annotations: unavailable
Test-set detection metrics: cannot be calculated without labels.
